In [1]:
import pandas as pd
import numpy as np
import torch
import joblib
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Device: {device}")

C:\Users\Riyad\AppData\Roaming\Python\Python310\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
C:\Users\Riyad\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 Device: cuda


In [2]:
test_df = pd.read_csv('C:/Users/Riyad/projects/fake_news/test_final.csv')
test_df['text'] = test_df['headline'].fillna('') + ' ' + test_df['content'].fillna('')

label2id = {'authentic': 0, 'fake': 1, 'ai_fake': 2}
id2label = {0: 'authentic', 1: 'fake', 2: 'ai_fake'}
test_df['label_id'] = test_df['label'].map(label2id)

print("Data loaded!")
print(f"Test: {len(test_df)}")

Data loaded!
Test: 2250


In [4]:
# SVM predictions 
svm_model = joblib.load('C:/Users/Riyad/projects/fake_news/svm_model.pkl')
svm_preds = svm_model.predict(test_df['text'].values)
svm_pred_ids = np.array([label2id[p] for p in svm_preds])

print("SVM predictions done!")
print(f"Shape: {svm_pred_ids.shape}")
print(f"Example: {svm_pred_ids[:5]}")

SVM predictions done!
Shape: (2250,)
Example: [1 1 1 2 1]


In [5]:
# BanglaBERT predictions with probability
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Load BanglaBERT
tokenizer = AutoTokenizer.from_pretrained("csebuetnlp/banglabert")
model = AutoModelForSequenceClassification.from_pretrained(
    "csebuetnlp/banglabert", num_labels=3)
model.load_state_dict(torch.load(
    'C:/Users/Riyad/projects/fake_news/banglabert_best.pt'))
model = model.to(device)
model.eval()

# Get probabilities
test_dataset = NewsDataset(
    test_df['text'].values,
    test_df['label_id'].values,
    tokenizer)
test_loader = DataLoader(test_dataset, batch_size=16)

bert_proba = []
bert_preds = []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        proba = torch.softmax(outputs.logits, dim=1)
        preds = outputs.logits.argmax(dim=1)
        bert_proba.extend(proba.cpu().numpy())
        bert_preds.extend(preds.cpu().numpy())

bert_proba = np.array(bert_proba)
bert_preds = np.array(bert_preds)

print("BanglaBERT probabilities done!")
print(f"Shape: {bert_proba.shape}")

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BanglaBERT probabilities done!
Shape: (2250, 3)


In [6]:


true_labels = test_df['label_id'].values

# Method 1: Majority Vote
ensemble_preds = []
for svm_pred, bert_pred in zip(svm_pred_ids, bert_preds):
    if svm_pred == bert_pred:
        ensemble_preds.append(svm_pred)
    else:
        # disagreement → BanglaBERT 
        ensemble_preds.append(bert_pred)

ensemble_preds = np.array(ensemble_preds)

# Results
svm_f1 = f1_score(true_labels, svm_pred_ids, average='macro')
bert_f1 = f1_score(true_labels, bert_preds, average='macro')
ensemble_f1 = f1_score(true_labels, ensemble_preds, average='macro')

print("="*45)
print("Ensemble Results")
print("="*45)
print(f"SVM F1:        {svm_f1*100:.2f}%")
print(f"BanglaBERT F1: {bert_f1*100:.2f}%")
print(f"Ensemble F1:   {ensemble_f1*100:.2f}%")
print("="*45)
print("\nDetailed Report:")
print(classification_report(
    true_labels, ensemble_preds,
    target_names=['authentic', 'fake', 'ai_fake']))

Ensemble Results
SVM F1:        92.00%
BanglaBERT F1: 91.72%
Ensemble F1:   91.72%

Detailed Report:
              precision    recall  f1-score   support

   authentic       0.86      0.92      0.89       750
        fake       0.90      0.84      0.87       750
     ai_fake       0.99      0.99      0.99       750

    accuracy                           0.92      2250
   macro avg       0.92      0.92      0.92      2250
weighted avg       0.92      0.92      0.92      2250



In [7]:
# Method 2: SVM 
ensemble_preds2 = []
for svm_pred, bert_pred in zip(svm_pred_ids, bert_preds):
    if svm_pred == bert_pred:
        ensemble_preds2.append(svm_pred)
    else:
        # disagreement → SVM 
        ensemble_preds2.append(svm_pred)

ensemble_preds2 = np.array(ensemble_preds2)
ensemble_f1_2 = f1_score(true_labels, ensemble_preds2, average='macro')

# Method 3: BanglaBERT probability + SVM vote
ensemble_preds3 = []
for svm_pred, bert_pred, bert_prob in zip(svm_pred_ids, bert_preds, bert_proba):
    if svm_pred == bert_pred:
        ensemble_preds3.append(svm_pred)
    else:
        # disagreement → confidence
        if max(bert_prob) > 0.7:
            ensemble_preds3.append(bert_pred)
        else:
            ensemble_preds3.append(svm_pred)

ensemble_preds3 = np.array(ensemble_preds3)
ensemble_f1_3 = f1_score(true_labels, ensemble_preds3, average='macro')

print("="*45)
print("All Ensemble Methods")
print("="*45)
print(f"SVM only:           {svm_f1*100:.2f}%")
print(f"BanglaBERT only:    {bert_f1*100:.2f}%")
print(f"Ensemble (BERT win): {ensemble_f1*100:.2f}%")
print(f"Ensemble (SVM win):  {ensemble_f1_2*100:.2f}%")
print(f"Ensemble (Confidence): {ensemble_f1_3*100:.2f}%")
print("="*45)

All Ensemble Methods
SVM only:           92.00%
BanglaBERT only:    91.72%
Ensemble (BERT win): 91.72%
Ensemble (SVM win):  92.00%
Ensemble (Confidence): 91.90%
